# Lab 2: Getting Data In, and Judging It

**DSA 405 · Week 2**

| | |
|---|---|
| **In class** | Friday, Aug 28 |
| **A2 due** | Thursday, Sep 3, 11:59 PM |
| **Also due Thu Sep 3** | **P1** Framing a Data Problem (separate handout) |
| **Files** | `wolfpack_dining_raw.csv`, `nc_schools_dirty.xlsx`, `permits_raleigh.json` |
| **Time** | ~25 min in class, ~55 min at home |

## Overview

Loading a file into `pandas` takes one line. This Lab covers the decisions that line
makes by default (types, headers, what counts as missing) and how to check those
decisions before relying on the result.

The main file through Week 4 is `wolfpack_dining_raw.csv`: 366 inspection records for
campus dining locations. The data is synthetic and public, with defects modeled on
real files.

In [ ]:
# ---------------------------------------------------------------------------
# DSA 405 setup
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np, requests, io

DATA = "https://raw.githubusercontent.com/jon-holt/DSA-405-Student/main/datasets/"
# DATA = "data/raw/"          # local users


def load(filename, kind="csv", **kw):
    """Read a class file whether DATA is a URL or a local folder."""
    path = DATA + filename
    if kind == "csv":
        return pd.read_csv(path, **kw)
    if kind == "excel":
        return pd.read_excel(path, **kw)
    if kind == "text":
        return requests.get(path, timeout=30).text if path.startswith("http") else open(path).read()
    if kind == "json":
        if path.startswith("http"):
            return requests.get(path, timeout=30).json()
        import json as _j
        return _j.load(open(path))
    raise ValueError(kind)


pd.set_option("display.width", 160)
print("pandas", pd.__version__)

---
# Part 1: Explore (in class)

## Task 1.1: First load, first inspection

A default read usually completes without error. Completing without error is not the
same as reading the file correctly.

In [ ]:
dining = load("wolfpack_dining_raw.csv")

print(dining.shape)
dining.head()

366 rows, 10 columns, and `.head()` shows nothing unusual. Check what dtype `pandas`
assigned each column:

In [ ]:
dining.dtypes

Nine of the ten columns came back as `object`, i.e. strings. Columns like `score` and
`seats` should be numeric but contain text somewhere, so `pandas` left them as strings.
That refusal is information about the file.

The one column that did convert, `unit_code`, should not have: unit codes are labels,
not quantities. The conversion has a cost:

In [ ]:
as_text = load("wolfpack_dining_raw.csv", dtype=str)

lost = as_text.unit_code.str.startswith("0").sum()
print(f"unit codes that begin with 0: {lost} of {len(as_text)}")
print("as loaded by default:", dining.unit_code.head(3).tolist())
print("as they are in the file:", as_text.unit_code.head(3).tolist())

31 of 366 codes begin with a zero, and the default read removed it: `0352` became
`352`, a different label. A join on that column would silently fail to match those 31
rows. The fix is one argument at load time: `dtype={"unit_code": str}`.

## Task 1.2: The profiling kit

Four methods cover most of what can be learned about a fresh file. Run them and read
the full output:

In [ ]:
dining.info()

In [ ]:
# value_counts with dropna=False is the honest version — NaN gets a row too
print(dining.category.value_counts(dropna=False).head(10))
print()
print("distinct category strings:", dining.category.nunique())

27 distinct category strings. Read the full list and estimate how many real categories
it represents. (`Coffee`, `COFFEE`, `coffee `, and `Coffee Shop` are variants of the
same value; Week 4 covers the repair.)

Next, profile a column that should be numeric:

In [ ]:
print(dining.score.value_counts(dropna=False).head(12))

The column contains numbers, and also several values that are not numbers. `.isna()`
detects almost none of them. A2 Task 2.2 addresses this directly.

## Task 1.3: Excel and JSON

CSV is the simplest case. Excel files are often formatted for human readers, and JSON
arrives as nested structure rather than rows. Each requires one extra loading decision.

In [ ]:
# the naive read
schools_naive = load("nc_schools_dirty.xlsx", "excel")
schools_naive.head()

The sheet is not yet a table: three rows of title text sit above the real headers.
Tell `read_excel` where the data starts:

In [ ]:
schools = load("nc_schools_dirty.xlsx", "excel", header=3)
print(schools.shape)
schools.head(3)

In [ ]:
# and it is THREE sheets, not one. sheet_name=None returns a dict of DataFrames.
all_sheets = load("nc_schools_dirty.xlsx", "excel", header=3, sheet_name=None)
for name, df in all_sheets.items():
    print(f"{name}: {df.shape}")

In [ ]:
# JSON: not a table at all until you make it one
permits = load("permits_raleigh.json", "json")
print(type(permits), "with keys:", list(permits.keys()))
print("records:", len(permits["results"]))
permits["results"][0]

The result is a dict of lists of dicts. `pandas` can flatten it, but choosing which
fields at which level is a decision; Week 11 covers it properly. For now: JSON files
arrive as structure, not as rows.

---
## Checkpoint: submit before leaving class

1. What did the default read do to `unit_code`, and exactly how many rows does it affect?
2. How many distinct `category` strings are there, and how many real categories do they
   appear to represent?
3. Name one column of `wolfpack_dining_raw.csv` that is not yet trustworthy, and why.

*Answers here.*

---
# Part 2: A2 (Loading & Profiling)

Graded. Four tasks.

## Task 2.1: The dtype audit

Load the dining file with default settings and audit what came back.

1. Report each column's dtype. State which columns should be numeric but are not, and
   which column converted but should not have.
2. Two columns have far more distinct values than they should. Name both, give the
   counts, and show two or three example values that explain the inflation.

In [ ]:
# your audit

*Name the two columns, the counts, and what inflates them.*

## Task 2.2: The missing-value census

`score`, `seats`, and `avg_ticket` all contain missing values, but almost none of them
are `NaN`. Using `value_counts(dropna=False)` on the **string** version of each column
(`dtype=str`), build a census: every distinct way "no value" is encoded, and how many times
each appears.

Then run `pd.to_numeric(..., errors="coerce")` on each column. Which missing-value
encodings survive as real numbers? Count them. Explain in two sentences why a sentinel
that survives conversion is more dangerous than one that becomes `NaN`. (Consider what
`.mean()` does with each.)

In [ ]:
# your census

*Census and two sentences here.*

## Task 2.3: The leading-zero repair

Demonstrate the damage, then the repair:

1. Count how many `unit_code` values lose a leading zero on a default read. Show one
   before/after pair.
2. Re-load with the correct `dtype` argument and verify the count of zero-leading codes.
3. One sentence: name something real that breaks when `0352` becomes `352`.

In [ ]:
# your repair

## Task 2.4: One table from the wild

`read_html` pulls every table off a web page at once. This is the course's first
scrape — and your first taste of a site pushing back: Wikipedia returns
`403 Forbidden` to anonymous scripts, so the starter cell identifies itself with an
honest User-Agent before asking. Why that matters is Week 8's whole topic.

1. Pick a Wikipedia page with a real table (a sport, a chart, a discography) that can
   be sanity-checked by eye. `pd.read_html(...)` returns a **list**; locate the target
   table in it.
2. Perform **one** cleaning action the table needs (drop a junk row, fix a header,
   convert a column) and state the row count before and after.
3. The graded centrepiece, in prose: judge the table's fitness for one specific
   purpose. Who or what is missing from it? Who decided what counts as a row? A table of
   "every #1 hit" contains decisions someone made; name one and say who it leaves out.

A paragraph that names something specific is worth more than three that say "the data
may be incomplete." 

In [ ]:
URL = "..."   # your Wikipedia page

# Wikipedia refuses anonymous scripts, so say who you are. Honest
# identification, not disguise — this is Week 8's topic in miniature.
UA = {"User-Agent": "DSA405-student-lab/1.0 (NC State class exercise)"}

# html = requests.get(URL, headers=UA, timeout=30).text
# tables = pd.read_html(io.StringIO(html))

*Fitness-for-purpose paragraph here.*

---
## AI use note

List any AI tools used and what they were used for. If none, write "none." One or two
sentences.

*Answer here.*

---
## Submitting

1. **Runtime > Restart runtime**, then **Run all**.
2. `File > Download > Download .ipynb`
3. Rename to `DSA405_002_FA26_A2_[yourUnityID].ipynb`
4. Upload to the **A2** space on Moodle.

The **Checkpoint** section is submitted separately to **Week 2 In-Class Activity**, before
the end of class on Friday. Due for A2: **Thursday, Sep 3, 11:59 PM**.